# 05 — The DLL Wrapper

`iwfm_io.dll` wraps the official IWFM Fortran DLL (`IWFM_C_x64.dll`,
~225 exported functions) with three classes:

- **`IWFMModel`** — open a model, query grid/heads/streams/budgets live
- **`IWFMBudget`** — standalone budget HDF reader
- **`IWFMZBudget`** — standalone zone-budget reader

**Requires: Windows x64** + the sample model. The DLL itself is fetched
on demand from the project's GitHub release assets — no manual install.

When do you need the DLL at all? Mostly for *live simulation state*.
Everything file-derived (grids, heads, budgets, hydrographs) is also
served DLL-free by `open_model()` — and several getters don't work in
the DLL's inquiry mode anyway (notes below). Prefer the adapter unless
you specifically need the DLL.

In [1]:
import sys

assert sys.platform == "win32", "the DLL wrapper requires Windows x64"

import os
from pathlib import Path


def find_sample_model():
    env = os.environ.get("IWFM_SAMPLE_MODEL")
    if env:
        return Path(env)
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / ".assets" / "sample_model"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("sample model not found — see notebook 01")


SAMPLE_MODEL = find_sample_model()
# IWFMModel wants the preprocessor MAIN input file (.IN), NOT the .bin
PREPROC_MAIN = SAMPLE_MODEL / "Preprocessor" / "PreProcessor_MAIN.IN"
SIM_MAIN = SAMPLE_MODEL / "Simulation" / "Simulation_MAIN.IN"
BEGIN, END = "10/01/1990_24:00", "09/30/2000_24:00"

## DLL version management

Multiple official DWR builds are supported side by side. Six builds are
published as `dll-<version>` release assets (sha256-verified on
download); `load_dll()` resolves in this order:

1. explicit `dll_path=` argument
2. explicit `version=` argument
3. `IWFM_DLL_VERSION` environment variable
4. `dlls/default_version.txt` in the repo
5. legacy auto-discovery

`~/.iwfm/dlls/` is searched at each version-based step, so
`download_dll()` once and every project on the machine finds it.

In [2]:
import iwfm_io

print("installed versions:", iwfm_io.dll.list_dll_versions())

# fetch a build if none is installed yet (no-op if already present)
if not iwfm_io.dll.list_dll_versions():
    iwfm_io.dll.download_dll("2025.0.1747")
    print("downloaded:", iwfm_io.dll.list_dll_versions())

installed versions: ['2015.0.1403', '2015.1.1273', '2015.3.1443', '2024.2.1594', '2025.0.1688', '2025.0.1747']


In [3]:
dll = iwfm_io.dll.load_dll()
print("IWFM version:  ", iwfm_io.dll.get_version(dll))
print("kernel version:", iwfm_io.dll.get_kernel_version(dll))

DLL does not export IW_GetLocationTypeID_Bypass -- skipping registration


IWFM version:   IWFM       : 2025.0.1747-054223d4
IWFM Kernel: 2025.0.107-8bd472ec
kernel version: 2025.0.107-8bd472ec


## Open the model

`is_for_inquiry=True` opens the model read-only against its existing
results (first open builds an inquiry cache, minutes on big models;
afterwards it's instant). Pin a DLL build per model with
`IWFMModel(..., dll_version="2024.2.1594")` — e.g. C2VSimFG v1.5 wants
its own version.

One caveat: the DLL resolves the model's relative output paths against
the **process working directory**, so we chdir to the Simulation folder
like IWFM itself would run.

In [4]:
NOTEBOOK_DIR = Path.cwd()
os.chdir(SIM_MAIN.parent)

m = iwfm_io.dll.IWFMModel(
    preprocessor_file=str(PREPROC_MAIN),
    simulation_file=str(SIM_MAIN),
    is_for_inquiry=True,
)
print(f"nodes={m.n_nodes} elements={m.n_elements} "
      f"layers={m.n_layers} subregions={m.n_subregions}")

DLL does not export IW_GetLocationTypeID_Bypass -- skipping registration


nodes=441 elements=400 layers=2 subregions=2


## Grid and stratigraphy queries

In [5]:
x, y = m.get_node_coordinates()
print(f"extent: X [{x.min():.0f}, {x.max():.0f}]  Y [{y.min():.0f}, {y.max():.0f}]")

for sid in m.get_subregion_ids():
    print(f"subregion {sid}: {m.get_subregion_name(sid)}")

gse = m.get_ground_surface_elevation()
print(f"GSE (ft): min={gse.min():.1f} max={gse.max():.1f} mean={gse.mean():.1f}")

extent: X [1804440, 1935672]  Y [14435520, 14566752]
subregion 1: Region1 (SR1)
subregion 2: Region2 (SR2)
GSE (ft): min=250.0 max=500.0 mean=490.0


In [6]:
# Aquifer parameters return (n_nodes, n_layers) arrays. On some models
# (including this one) the inquiry-mode re-read hits a DLL bug — the
# DLL-free adapter always works instead (docs/DLL_INQUIRY_MODE_LIMITS.md).
try:
    kh = m.get_aquifer_horizontal_k()
    for lyr in range(m.n_layers):
        print(f"layer {lyr + 1} Kh: [{kh[:, lyr].min():.3f}, "
              f"{kh[:, lyr].max():.3f}]")
except Exception as exc:
    print("aquifer parameters unavailable in inquiry mode "
          f"(known DLL limitation): {str(exc)[:90]}")

aquifer parameters unavailable in inquiry mode (known DLL limitation): IWFM error (-1): * FATAL:
*   Node ID 1 is listed more than once for initial groundwater h


## Head time series and depth to water

In [7]:
dates, heads = m.get_gw_heads_for_layer(layer=1, begin_date=BEGIN, end_date=END)
print(f"layer 1: {len(dates)} timesteps x {heads.shape[0]} nodes, "
      f"head range [{heads.min():.1f}, {heads.max():.1f}] ft")

dtw = gse - heads[:, -1]
print(f"depth to water at last step: min={dtw.min():.1f} "
      f"max={dtw.max():.1f} mean={dtw.mean():.1f} ft")

layer 1: 3653 timesteps x 441 nodes, head range [126.4, 537.1] ft
depth to water at last step: min=-35.0 max=362.4 mean=150.4 ft


### DLL dates are Excel serial numbers

The DLL returns times as floating-point days since 1899-12-30; convert
with `excel_date_to_datetime` from the plots package.

In [8]:
from iwfm_io.plots import excel_date_to_datetime

print(dates[:3], "->", [excel_date_to_datetime(d) for d in dates[:3]])

[33147. 33148. 33149.] -> [datetime.datetime(1990, 10, 1, 0, 0), datetime.datetime(1990, 10, 2, 0, 0), datetime.datetime(1990, 10, 3, 0, 0)]


## Stream network

In [9]:
print(f"stream nodes: {m.n_stream_nodes}, reaches: {m.n_reaches}")
elevs = m.get_stream_bottom_elevations()
print(f"bottom elevations: [{elevs.min():.1f}, {elevs.max():.1f}] ft")
for rid in m.get_reach_ids():
    nodes = m.get_reach_stream_nodes(rid)
    print(f"reach {rid}: stream nodes {nodes.min()}-{nodes.max()}")

stream nodes: 23, reaches: 3
bottom elevations: [260.0, 300.0] ft
reach 2: stream nodes 17-23
reach 3: stream nodes 1-10
reach 1: stream nodes 11-16


## Hydrographs

`get_hydrograph` masks uninitialized buffer entries (`date > 0`) — the
raw DLL call reports the requested-window size, not the valid count.

In [10]:
hyd_types = m.get_hydrograph_type_list()
print("hydrograph types:", [t["name"] for t in hyd_types])

ht = hyd_types[0]
ids = m.get_hydrograph_ids(ht["location_type"])
# hydrographs come back at their native output interval (here daily);
# the DLL's own re-sampling strides fixed day counts, so resample in
# pandas instead when a coarser series is wanted
dates_h, vals = m.get_hydrograph(
    hyd_type=ht["location_type"], index=int(ids[0]), layer=1,
    begin_date=BEGIN, end_date=END, interval="1DAY")
valid = dates_h > 0
print(f"{ht['name']}[{ids[0]}]: {valid.sum()} valid steps, "
      f"values [{vals[valid].min():.2f}, {vals[valid].max():.2f}]")
import pandas as pd
monthly = pd.Series(vals[valid], index=m._excel_dates_to_index(dates_h[valid])).resample("MS").mean()
print(f"monthly means: {len(monthly)} months")

hydrograph types: ['Groundwater hydrograph', 'Groundwater hydrograph at node and layer', 'Subsidence hydrograph', 'Tile drain hydrograph', 'Stream hydrograph (flow)']
Groundwater hydrograph[1]: 3653 valid steps, values [282.00, 302.31]
monthly means: 120 months


## Budgets through the model

In [11]:
budgets = m.get_budget_list()
print("available budgets:", [b["name"] for b in budgets])

bt = budgets[0]["budget_type"]
cols = m.get_budget_column_titles(bt, location=1)
print("first columns:", cols[:4])

result = m.get_budget_timeseries(
    budget_type=bt, location=1, columns=[1, 2, 3],
    begin_date=BEGIN, end_date=END, interval="1MON")
print(f"{len(result['dates'])} timesteps x {result['values'].shape[1]} columns")

available budgets: ['Groundwater budget', 'Stream node budget', 'Stream reach budget', 'Land and water use budget', 'Root zone budget', 'Unsaturated zone budget', 'Lake budget', 'Small watershed budget']
first columns: ['Percolation', 'Beginning Storage (+)', 'Ending Storage (-)', 'Deep Percolation (+)']
120 timesteps x 3 columns


In [12]:
m.close()

## `IWFMBudget` — standalone budget HDF reader

Opens a budget HDF directly (no model needed). Same engine the IWFM
Budget post-processor uses.

In [13]:
RESULTS_DIR = SAMPLE_MODEL / "Results"

with iwfm_io.dll.IWFMBudget(str(RESULTS_DIR / "GW.hdf")) as bud:
    print(f"locations: {bud.get_location_names()}")
    print(f"timesteps: {bud.n_timesteps}")
    values = bud.get_values(location=1, columns=[1, 2, 3],
                            begin_date=BEGIN, end_date=END, interval="1MON")
    print(f"values shape (time + 3 columns): {values.shape}")

DLL does not export IW_GetLocationTypeID_Bypass -- skipping registration


locations: ['Region1 (SR1)', 'Region2 (SR2)', 'ENTIRE MODEL AREA']
timesteps: 3653
values shape (time + 3 columns): (120, 4)


## `IWFMZBudget` — standalone zone-budget reader

In [14]:
zhdf = RESULTS_DIR / "GW_ZBud.hdf"
if zhdf.exists():
    with iwfm_io.dll.IWFMZBudget(str(zhdf)) as zbud:
        zones = zbud.get_zone_list()
        if zones is None or len(zones) == 0:
            print("no zone list published by the DLL for this model "
                  "(known sample-model limitation) — the DLL-free "
                  "read_zbudget_hdf in notebook 03 works instead")
        else:
            print("zones:", zones)
else:
    print("GW_ZBud.hdf not present — run ZBudget to generate it")

DLL does not export IW_GetLocationTypeID_Bypass -- skipping registration


no zone list published by the DLL for this model (known sample-model limitation) — the DLL-free read_zbudget_hdf in notebook 03 works instead


## Type-ID enums

The DLL publishes its type constants at runtime;
`load_all_type_ids` populates enum classes from them.

In [15]:
iwfm_io.dll.load_all_type_ids(dll)
for cls_name in ("BudgetTypeID", "LocationTypeID"):
    cls = getattr(iwfm_io.dll, cls_name)
    members = {k: v for k, v in vars(cls).items() if not k.startswith("_")}
    print(f"{cls_name}: {dict(list(members.items())[:4])} ...")

BudgetTypeID: {'GW': 3001, 'RootZone': 4002, 'LWU': 4001, 'NonPondedCrop_RZ': 4004} ...
LocationTypeID: {'Node': 8, 'Element': 2, 'Subregion': 4, 'Zone': 7} ...


In [16]:
os.chdir(NOTEBOOK_DIR)

## Inquiry-mode limitations — and their DLL-free escapes

In inquiry mode the DLL never instantiates RootZone or the dynamic
GW/stream components, so some getters fail by design. Every one has a
file-based alternative through `open_model()`:

| DLL limitation | DLL-free path |
|----------------|---------------|
| aquifer-parameter re-read bug (layout-sensitive) | parsed from GW main (`read_gw_main`) |
| tile drains, bypasses, land use, supply/shortage, pumping | adapter serves them from files |
| stream–GW exchange arrays not allocated | stream node budget HDF |
| zone list empty on some models | `read_zbudget_hdf` / `get_zbudget_timeseries` |

Full root-cause analyses: `docs/DLL_INQUIRY_MODE_LIMITS.md`.